### Note on data source
Unlike `loan_default`, I couldn't find/verify a stable direct-CSV mirror for this dataset (the GitHub copies I found disallow raw fetches or are zipped). Since it's a real Kaggle dataset, the most reliable way to pull it is via `kagglehub`, which uses your local Kaggle API credentials (`~/.kaggle/kaggle.json`).

If you'd rather use a specific mirror URL you already trust, skip the next cell and just set `CONFIG["data_path"]` to that URL instead (everything else below is unaffected).

In [1]:
import sys
sys.path.insert(0, "..")
from src.preprocessing import run_pipeline


In [2]:
import kagglehub
import shutil
import os

# Download the dataset (it will go to cache by default)
kaggle_dir = kagglehub.dataset_download("shebrahimi/financial-distress")

# Your desired destination path
desired_path = r"C:\Users\Offic\Downloads\pima_template_project_v3_updated (1)\Datasets\finance\financial_distress\raw"  # Change this to your path

# Create destination directory if it doesn't exist
os.makedirs(desired_path, exist_ok=True)

raw_csv_path = os.path.join(desired_path, "Financial Distress.csv")

# Copy all files from cache to desired location
for file_name in os.listdir(kaggle_dir):
    source_file = os.path.join(kaggle_dir, file_name)
    dest_file = os.path.join(desired_path, file_name)
    if os.path.isfile(source_file):
        shutil.copy2(source_file, dest_file)

print(f"Dataset copied to: {desired_path}")


Dataset copied to: C:\Users\Offic\Downloads\pima_template_project_v3_updated (1)\Datasets\finance\financial_distress\raw


In [3]:
CONFIG = {
    "dataset_name": "financial_distress",
    "domain": "finance",
    "source": "Kaggle: shebrahimi/financial-distress",

    "data_path": raw_csv_path,   # local path from kagglehub download above (or swap in a mirror URL)
    "column_names": None,        # file already has a header row
    "target": "Financial Distress",

    # 'Company' is a per-firm identifier (analogous to LoanID) -- not a feature.
    # Kept out of drop_columns is 'Time' (the fiscal-period index, 1-14): this is
    # panel data (multiple rows per company across periods), and this pipeline's
    # protocol does a plain row-level split rather than a company-grouped one --
    # the same simplification applied to every other dataset in this study. 'Time'
    # is kept as an ordinary numerical feature, consistent with how ordinal/period
    # columns (e.g. LoanTerm in loan_default) were treated elsewhere.
    "drop_columns": ["Company"],

    # The raw target is a continuous distress score (range roughly -8.6 to 128.4).
    # The published convention for this exact dataset (used across the original
    # author's own notebooks and most downstream studies) is: healthy (0) if the
    # score is > -0.50, distressed (1) otherwise. This is a fixed, documented rule,
    # not a statistic fit on the data, so it's safe to apply before the split.
    "target_binarize": {"threshold": -0.50, "positive_if": "leq"},
    "target_mapping": None,

    # No sentinel-zero issue here (financial ratios/scores are legitimately
    # negative or zero). numerical_cols/categorical_cols are left unset:
    # x1-x83 and Time are all numeric dtype already (a few, e.g. x80/x82/x83,
    # are integer-coded with high cardinality -- treated as numerical rather
    # than one-hot encoded, consistent with how the dataset is handled in the
    # published literature on it).
    "zero_as_missing_cols": None,

    "random_state": 42,
    "test_size": 0.20,

    "imputation_strategy_num": "median",
    "imputation_strategy_cat": "most_frequent",
    "scaler": "StandardScaler",

    # Distressed (1) is a small minority (~4% of rows) after binarization --
    # SMOTE on the training set only, consistent with the other datasets.
    "balancing": "SMOTE",

    "high_missingness_warn_threshold": 0.40,

    "feature_descriptions": {
        "Time": "Fiscal period index for the company (1-14, anonymized time unit)",
        "Financial Distress": "Continuous distress score; binarized to 0=healthy (>-0.50), 1=distressed (<=-0.50)",
        # x1-x83: anonymized financial/non-financial ratios and indicators as
        # provided by the dataset author; no public data dictionary exists
        # mapping them to named accounting variables.
    },

    "datasets_root": "../Datasets",
}


In [4]:
results = run_pipeline(CONFIG)


CONFIG: {'dataset_name': 'financial_distress', 'domain': 'finance', 'source': 'Kaggle: shebrahimi/financial-distress', 'data_path': 'C:\\Users\\Offic\\Downloads\\pima_template_project_v3_updated (1)\\Datasets\\finance\\financial_distress\\raw\\Financial Distress.csv', 'column_names': None, 'target': 'Financial Distress', 'drop_columns': ['Company'], 'target_binarize': {'threshold': -0.5, 'positive_if': 'leq'}, 'target_mapping': None, 'zero_as_missing_cols': None, 'random_state': 42, 'test_size': 0.2, 'imputation_strategy_num': 'median', 'imputation_strategy_cat': 'most_frequent', 'scaler': 'StandardScaler', 'balancing': 'SMOTE', 'high_missingness_warn_threshold': 0.4, 'feature_descriptions': {'Time': 'Fiscal period index for the company (1-14, anonymized time unit)', 'Financial Distress': 'Continuous distress score; binarized to 0=healthy (>-0.50), 1=distressed (<=-0.50)'}, 'datasets_root': '../Datasets'}

------------------------------------------------------------
2. DATASET LOADING
